
# Unit 2 — Advanced Visualization & Storytelling (UE24CS342AA9)
## Activity Notebook: Build an Interactive Retail Sales Dashboard

### Scenario

You have just joined the analytics team of **UrbanCart**, a mid-sized retail chain with stores across
three regions. Regional managers currently receive a static monthly PDF report and complain that:

1. They can't drill into *why* a number moved without emailing the analytics team.
2. Comparing stores or products side by side means flipping between many separate PDF pages.
3. Spotting a sudden dip in a specific product/store is slow and easy to miss.

**Your job:** apply the interaction techniques from the lecture — dynamic queries, coordinated views
& brushing, table lens, focus+context drill-down, and single-screen dashboard design — to build an
**interactive exploration tool** that solves these three complaints.

### How this notebook is organized

- A sample dataset is generated for you (Task 0) — don't skip running it.
- Each task states the **real-world usage**: which complaint above it solves, and why the technique
  fits.
- Tasks give you a **scaffold** (imports, helper stubs, expected output) — **you write the core logic**
  marked with `# TODO`. Do not just copy the Demo notebook's code verbatim — the data shape, column
  names and the exact question being asked are different here, so you will need to adapt the pattern,
  not paste it.


> **Grading-relevant tip:** every `# TODO` cell tells you what the final variable/plot must be named
> or show — read it before coding, since later cells depend on it.



## Task 0 — Load the Sample Dataset (provided)

Run the cells below as-is. This generates `sales_df`: 24 months of daily-aggregated sales across
6 stores (3 regions), 10 products (4 categories), including revenue, profit and customer ratings —
enough structure to support every technique you'll build below.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

import plotly.express as px
import plotly.graph_objects as go

import ipywidgets as widgets
from ipywidgets import interact, HBox, VBox

sns.set_theme(style="whitegrid")
np.random.seed(7)
print("Libraries loaded.")


In [ ]:

# --- Sample dataset generator: UrbanCart retail sales (provided, do not need to modify) ---
stores = pd.DataFrame({
    "store_id":   ["S01", "S02", "S03", "S04", "S05", "S06"],
    "store_city": ["Bengaluru", "Mysuru", "Mumbai", "Pune", "Delhi", "Jaipur"],
    "region":     ["South", "South", "West", "West", "North", "North"],
})

products = pd.DataFrame({
    "product":  ["Wireless Earbuds", "Smartwatch", "Laptop Sleeve", "USB-C Hub",
                 "Yoga Mat", "Dumbbell Set", "Running Shoes", "Track Jacket",
                 "Air Fryer", "Blender"],
    "category": ["Electronics", "Electronics", "Electronics", "Electronics",
                 "Fitness", "Fitness", "Apparel", "Apparel",
                 "Home", "Home"],
    "unit_price": [1499, 4999, 899, 1299, 799, 2499, 3499, 1999, 3999, 2299],
})

months = pd.date_range("2023-01-01", periods=24, freq="MS")

rows = []
for _, s in stores.iterrows():
    store_scale = np.random.uniform(0.7, 1.4)          # some stores just sell more
    for _, p in products.iterrows():
        product_scale = np.random.uniform(0.6, 1.6)
        trend = np.random.normal(0.01, 0.01)             # slow month-over-month growth/decline
        for i, m in enumerate(months):
            seasonal = 1 + 0.25 * np.sin(2 * np.pi * (m.month / 12))   # yearly seasonality
            units = max(0, np.random.poisson(15 * store_scale * product_scale * seasonal * (1 + trend) ** i))
            revenue = units * p["unit_price"]
            profit_margin = np.random.uniform(0.12, 0.35)
            profit = revenue * profit_margin
            rating = np.clip(np.random.normal(4.1, 0.4), 1, 5)
            # NOTE: use bracket indexing (p["product"]), not p.product -- pandas Series already has
            # a built-in .product() method, so attribute access on a column literally named "product"
            # silently returns that method instead of your data. This is a common real-world gotcha.
            rows.append((m, s["store_id"], s["store_city"], s["region"], p["product"], p["category"],
                         p["unit_price"], units, revenue, profit, round(rating, 1)))

sales_df = pd.DataFrame(rows, columns=[
    "month", "store_id", "store_city", "region", "product", "category",
    "unit_price", "units_sold", "revenue", "profit", "avg_rating"
])
for _col in ["units_sold", "revenue", "profit"]:
    sales_df[_col] = sales_df[_col].astype(float)  # float, so the anomaly injection below can scale them

# Inject one deliberate "anomaly" for Task 4 (a real dip a manager would want to investigate)
mask = (sales_df.store_id == "S03") & (sales_df["product"] == "Smartwatch") & (sales_df.month == "2024-06-01")
sales_df.loc[mask, ["units_sold", "revenue", "profit"]] = sales_df.loc[mask, ["units_sold", "revenue", "profit"]] * 0.15

print(sales_df.shape)
sales_df.head()


**Reflection (answer in a sentence or two, in this markdown cell):**
Before writing any code, look at the columns above. Which columns would you use as *filters* for
dynamic querying, and which would you use to *link* views together for brushing? Write your answer
here.

_Your answer:_
- **Filters for dynamic querying:** Continuous numeric attributes such as `unit_price`, `units_sold`, `revenue`, and `profit` (ideal for range sliders), along with categorical attributes like `region` and `category` (for dropdown or checkbox filtering).
- **Linking columns for brushing:** Entity keys that persist across multiple tables and views, specifically `store_id` (or `store_city`) to link store-level performance, `product` / `category` to link catalog metrics, and `month` to link temporal timelines.



## Task 1 — Static vs. Interactive: Which Report Actually Answers the Manager's Question?

**Real-world usage:** Complaint #1 ("flipping through PDF pages"). A regional manager asks:
*"Show me total revenue by product, and let me check exactly how much any bar is worth without
squinting at a printed axis."*

**Why this technique:** A static bar chart is fine for the *headline* number, but it can't answer a
precise follow-up question live — that needs a hover-enabled interactive chart (Slide 4-8).

### Your task
1. Build a **static** `matplotlib`/`seaborn` bar chart of total revenue per `product`, sorted
   descending.
2. Build the **same** chart as an **interactive** `plotly` bar chart where hovering shows the exact
   revenue figure and the category of that product.
3. In the markdown cell below, state in 1-2 sentences: for *this specific* manager request, which
   version wins, and why (tie your answer to target-audience / data-story / ROI from the slide).


In [ ]:
# --- 1a. STATIC chart ---
# TODO:
#   - Group sales_df by "product", sum "revenue"
#   - Sort descending
#   - Plot a horizontal or vertical bar chart with matplotlib/seaborn
#   - Title it clearly

revenue_by_product = sales_df.groupby("product")["revenue"].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=revenue_by_product.values / 1e6, y=revenue_by_product.index, palette="Blues_r")
plt.title("Total Revenue by Product (Static Overview)", fontsize=13, fontweight="bold")
plt.xlabel("Total Revenue (Million INR)")
plt.ylabel("Product")
plt.tight_layout()
plt.show()


In [ ]:
# --- 1b. INTERACTIVE chart ---
# TODO:
#   - Use plotly.express.bar on the SAME grouped data
#   - Add a hover field for "category" (hint: you'll need category alongside revenue -- think about
#     what columns your groupby needs to keep)
#   - Title it clearly

grouped_prod = sales_df.groupby(["product", "category"], as_index=False)["revenue"].sum()
grouped_prod = grouped_prod.sort_values("revenue", ascending=False)

fig = px.bar(
    grouped_prod,
    x="product",
    y="revenue",
    color="category",
    hover_name="product",
    hover_data={"revenue": ":,.2f", "category": True, "product": False},
    labels={"revenue": "Total Revenue (INR)", "product": "Product", "category": "Category"},
    title="Total Revenue by Product (Interactive Drill-Down with Exact Revenue & Category)"
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()


**Your 1-2 sentence answer:**

For this specific manager request, the interactive Plotly version clearly wins because it provides instant, zero-effort epistemic inspection: hovering reveals the exact revenue figure and product category on demand without requiring axis squinting or page flipping. This preserves cognitive flow and directly fulfills the manager's target-audience need for immediate precision while maintaining an uncluttered visual narrative.



## Task 2 — Dynamic Queries: Let Managers Filter Without Asking You

**Real-world usage:** Regional managers want to self-serve answer questions like *"which
products, in what price range, sold above X units last month?"* without emailing the analytics team
every time (Complaint #1 again, but for filtering rather than just reading).

**Why this technique:** Dynamic queries (double-ended range sliders) let the pattern emerge from the
noise live, with the response updating in real time (Slide 23).

### Your task
Build **two** linked `ipywidgets` range sliders:
- `price_slider` — filters on `unit_price`
- `units_slider` — filters on `units_sold`

Write a function `dynamic_filter(price_range, units_range)` that:
1. Filters `sales_df` to rows where `unit_price` is inside `price_range` **and** `units_sold` is
   inside `units_range`.
2. Prints how many rows match out of the total.
3. Shows a scatter plot of `unit_price` vs `units_sold`, with matching rows highlighted in a
   distinct color against all other rows in gray (same pattern as the Demo notebook's dynamic query
   cell — but you must build the filter condition and the plot yourself for these two columns).

Wire it up with `interact()`.


In [ ]:
# TODO: create price_slider (FloatRangeSlider) spanning sales_df.unit_price min..max
price_slider = widgets.FloatRangeSlider(
    value=[float(sales_df.unit_price.min()), float(sales_df.unit_price.max())],
    min=float(sales_df.unit_price.min()),
    max=float(sales_df.unit_price.max()),
    step=100.0,
    description="Unit Price:",
    continuous_update=False,
    layout=widgets.Layout(width="500px")
)

# TODO: create units_slider (FloatRangeSlider or IntRangeSlider) spanning sales_df.units_sold min..max
units_slider = widgets.IntRangeSlider(
    value=[int(sales_df.units_sold.min()), int(sales_df.units_sold.max())],
    min=int(sales_df.units_sold.min()),
    max=int(sales_df.units_sold.max()),
    step=1,
    description="Units Sold:",
    continuous_update=False,
    layout=widgets.Layout(width="500px")
)

def dynamic_filter(price_range, units_range):
    # TODO: filter sales_df using both ranges
    mask = (
        (sales_df.unit_price >= price_range[0]) & (sales_df.unit_price <= price_range[1]) &
        (sales_df.units_sold >= units_range[0]) & (sales_df.units_sold <= units_range[1])
    )
    filtered = sales_df[mask]

    # TODO: print "X / Y rows match"
    print(f"{len(filtered):,} / {len(sales_df):,} rows match ({len(filtered)/len(sales_df)*100:.1f}%)")

    # TODO: scatter plot: all rows in gray, filtered rows highlighted in color
    plt.figure(figsize=(9, 5))
    plt.scatter(sales_df.unit_price, sales_df.units_sold, c="lightgray", alpha=0.4, s=25, label="All Records")
    if len(filtered) > 0:
        plt.scatter(filtered.unit_price, filtered.units_sold, c="#1f77b4", alpha=0.8, s=30, label="Matching Records")
    plt.xlabel("Unit Price (INR)")
    plt.ylabel("Units Sold")
    plt.title(f"Dynamic Query: Price [{price_range[0]:.0f}, {price_range[1]:.0f}] & Units [{units_range[0]}, {units_range[1]}]")
    plt.legend()
    plt.tight_layout()
    plt.show()

# TODO: call interact(dynamic_filter, price_range=price_slider, units_range=units_slider)
interact(dynamic_filter, price_range=price_slider, units_range=units_slider)


**Reflection:** The slide notes dynamic queries work well "up to ~100,000 points with five or fewer
sliders." `sales_df` has far fewer rows. If UrbanCart grew to 500 stores and this dataset became 50x
larger, would two sliders still be enough, or would you need a different technique from the lecture?
Name one.

_Your answer:_
Two raw sliders alone would no longer be sufficient because rendering 50x more raw points causes severe visual overplotting (occlusion) and client-side interaction lag exceeding the 100ms perceptual responsiveness threshold. We would need **binned aggregation / 2D density rasterization (e.g., Datashader or hexbin aggregation)** or **faceted hierarchical brushing on pre-computed data cubes (OLAP)** as covered in the lecture, allowing the system to update pre-aggregated visual summaries rapidly before drilling down into raw records.



## Task 3 — Coordinated Views & Cross-View Brushing: Compare Stores Without Flipping Pages

**Real-world usage:** Complaint #2 — "comparing stores means flipping between many PDF pages."
A manager wants to click one store in an overview chart and instantly see that store's category
breakdown and its monthly trend, without losing sight of how it compares to the others.

**Why this technique:** Coordinated multiple views + brushing (Slide 24) — selecting a subset in one
view highlights the same records everywhere else, replacing the working-memory cost of flipping pages.

### Your task
Build **three linked views** for a store the user selects from a dropdown:
1. **Overview bar chart** — total revenue per store (all 6 stores), with the selected store's bar in
   a distinct color and every other store's bar in gray (this is the "muted unselected categories"
   trick from the slide).
2. **Category breakdown** — a pie or bar chart of revenue by `category`, filtered to the selected
   store only.
3. **Monthly trend line** — revenue by `month`, filtered to the selected store only.

Wrap all three in one function driven by a single `ipywidgets.Dropdown` of `store_id` values, and lay
the three charts out together (e.g. with `plt.subplots` for 1+2+3, or separate `display()` calls).


In [ ]:
store_overview = sales_df.groupby("store_id").revenue.sum().reset_index()

def store_coordinated_view(selected_store):
    # TODO 1: bar chart of store_overview, selected store highlighted, others muted gray
    # TODO 2: category breakdown pie/bar for selected_store only
    # TODO 3: monthly revenue trend line for selected_store only
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

    # 1. Overview bar chart
    colors = ["#1f77b4" if s == selected_store else "#d3d3d3" for s in store_overview.store_id]
    axes[0].bar(store_overview.store_id, store_overview.revenue / 1e6, color=colors)
    axes[0].set_title(f"All Stores Overview (Highlight: {selected_store})", fontweight="bold")
    axes[0].set_xlabel("Store ID")
    axes[0].set_ylabel("Total Revenue (Million INR)")

    # Filter data for selected store
    df_store = sales_df[sales_df.store_id == selected_store]

    # 2. Category breakdown
    cat_rev = df_store.groupby("category")["revenue"].sum()
    axes[1].pie(cat_rev, labels=cat_rev.index, autopct="%1.1f%%", startangle=140, colors=sns.color_palette("Set2"))
    axes[1].set_title(f"{selected_store}: Revenue by Category", fontweight="bold")

    # 3. Monthly revenue trend line
    monthly_rev = df_store.groupby("month")["revenue"].sum().reset_index()
    axes[2].plot(monthly_rev["month"], monthly_rev["revenue"] / 1e3, marker="o", color="#e6550d", linewidth=2)
    axes[2].set_title(f"{selected_store}: Monthly Revenue Trend", fontweight="bold")
    axes[2].set_xlabel("Month")
    axes[2].set_ylabel("Revenue ('000 INR)")
    axes[2].tick_params(axis="x", rotation=45)

    plt.tight_layout()
    plt.show()

# TODO: wire this up with interact() using a Dropdown of stores["store_id"] values
store_dropdown = widgets.Dropdown(
    options=sorted(stores["store_id"].unique()),
    value="S01",
    description="Store ID:"
)
interact(store_coordinated_view, selected_store=store_dropdown)


**Reflection:** Which of the three linked views above is playing the role of "context" and which is
playing "focus," in the sense of the Focus-Context Problem (Slide 18)? Is that the same distinction as
brushing, or a different but related idea? Explain briefly.

_Your answer:_
View 1 (All Stores Overview bar chart) plays the role of **context**, anchoring the global frame of reference so the manager understands where the selected store stands relative to the entire chain. Views 2 and 3 (Category breakdown and Monthly trend) play the role of **focus**, delivering high-resolution detail for the selected store. This is a related but distinct concept from brushing: **brushing** is the *interaction mechanism* (selecting a subset in one view to highlight corresponding elements in others), whereas **Focus+Context** is the overarching *information architecture paradigm* that ensures local inspection does not destroy global orientation.



## Task 4 — Focus + Context Drill-Down: Investigate the Dip

**Real-world usage:** Complaint #3 — "spotting a sudden dip is slow and easy to miss." One product at
one store has a real anomaly hidden in `sales_df` (planted in Task 0). A manager needs to find it and
drill in **without losing the overview**, using the *cheapest* interaction tier that still answers the
question (Drill-Down Cost Hierarchy, Slide 17).

### Your task
**Part A — find the anomaly.**
Write code that computes, for every `(store_id, product)` pair, the ratio of the *minimum* month's
`units_sold` to the *median* month's `units_sold` across the 24 months. Sort ascending and show the
top 5 most extreme drops. (Hint: `groupby(["store_id","product"]).units_sold.agg(...)` with a custom
function, or compute `min` and `median` separately and combine.)

**Part B — build the drill-down.**
Using **hover** (cheap tier) on an overview chart of all `(store_id, product)` trend lines, let the
user identify the anomalous line. Then, given the `store_id`/`product` you found in Part A, plot its
monthly trend **alongside** a faint gray context line showing the *average* trend across all
stores/products for the same product — so the dip is visible with its context still present (not a
full window replacement).


In [ ]:
# --- Part A: find the anomaly ---
# TODO: compute min/median ratio per (store_id, product) and show the 5 most extreme

anomaly_df = sales_df.groupby(["store_id", "product"])["units_sold"].agg(
    min_units="min",
    median_units="median"
).reset_index()
anomaly_df["ratio"] = anomaly_df["min_units"] / anomaly_df["median_units"]
anomaly_ranking = anomaly_df.sort_values("ratio").head(5).reset_index(drop=True)

print("Top 5 most extreme drops (min / median ratio):")
display(anomaly_ranking)


In [ ]:
# --- Part B: drill-down with context preserved ---
# TODO 1: build an overview plotly line chart of units_sold over month, one line per (store_id, product)
#         combination, with hover showing store_id + product (cheap-tier epistemic action)

sales_df["store_product"] = sales_df["store_id"] + " - " + sales_df["product"]

fig_overview = px.line(
    sales_df,
    x="month",
    y="units_sold",
    color="product",
    line_group="store_id",
    hover_name="store_product",
    hover_data={"store_id": True, "product": True, "units_sold": True, "store_product": False},
    title="Overview: Monthly Units Sold Across All Store-Product Combinations (Hover to Inspect)"
)
fig_overview.update_traces(opacity=0.35)
fig_overview.show()

# TODO 2: using the (store_id, product) pair you identified as the top anomaly in Part A,
#         plot its monthly units_sold trend (focus) together with the average trend for that SAME
#         product across all other stores (context), on one shared chart

top_anom = anomaly_ranking.iloc[0]
anom_store = top_anom["store_id"]
anom_product = top_anom["product"]

focus_data = sales_df[(sales_df.store_id == anom_store) & (sales_df["product"] == anom_product)].sort_values("month")
context_data = sales_df[(sales_df.store_id != anom_store) & (sales_df["product"] == anom_product)].groupby("month")["units_sold"].mean().reset_index()

plt.figure(figsize=(10, 4.5))
plt.plot(context_data["month"], context_data["units_sold"], label=f"Context: Other Stores Avg ({anom_product})", color="gray", linestyle="--", linewidth=2, alpha=0.7)
plt.plot(focus_data["month"], focus_data["units_sold"], label=f"Focus: {anom_store} ({anom_product})", color="#d62728", marker="o", linewidth=2.5)
plt.title(f"Focus + Context Drill-Down: Monthly Sales of {anom_product} at {anom_store} vs. Chain Average", fontsize=12, fontweight="bold")
plt.xlabel("Month")
plt.ylabel("Units Sold")
plt.legend()
plt.tight_layout()
plt.show()


**Reflection:** Which interaction-cost tier (eye fixation / hover / click-to-open panel / window
replacement) did you use to first *spot* the anomaly, and which did you use to *investigate* it? Was
using two different tiers the right call, per the lecture's design rule?

_Your answer:_
To first *spot* the anomaly across the entire store/product catalog, we used the **hover tier** (a low-cost epistemic action of ~100–300ms where scrubbing the mouse reveals store and product identities without interrupting the cognitive overview). To *investigate* it, we used a **focus+context overlay panel** (a shared visual workspace comparing the focal series with the normative baseline). Employing two distinct tiers strictly adheres to the lecture's Drill-Down Epistemic Cost Hierarchy: broad scanning is handled via minimal-cost visual/hover actions, while deeper analytical focus is reserved for confirmed exceptions without triggering disruptive window replacements.



## Task 5 — Table Lens: A Scannable Product Performance Table

**Real-world usage:** A manager wants a single table listing every product's total revenue, total
profit, and average rating across all stores — but a plain spreadsheet of 10 rows x many numeric
columns is still hard to compare at a glance across all three metrics simultaneously.

**Why this technique:** Table Lens (Slide 27) maps numeric columns to in-cell bars, so magnitude
differences are visible pre-attentively while the table stays sortable/readable as text.

### Your task
1. Build a summary DataFrame: one row per `product`, columns for total `revenue`, total `profit`, and
   mean `avg_rating`.
2. Sort it by total revenue, descending.
3. Style it as a Table Lens using `.style.bar(...)` — a different bar color per numeric column, as in
   the Demo notebook — and add a meaningful `.set_caption(...)`.
4. Below the table, add **one more cell**: sort the *same* summary table by `profit` instead of
   `revenue`, and re-render the styled table. Do the top rows change order? Note what that tells a
   manager about revenue vs. profitability.


In [ ]:
# TODO: build product_summary with columns: product, revenue (sum), profit (sum), avg_rating (mean)
product_summary = sales_df.groupby("product").agg(
    revenue=("revenue", "sum"),
    profit=("profit", "sum"),
    avg_rating=("avg_rating", "mean")
).reset_index()

# TODO: sort descending by revenue
product_summary = product_summary.sort_values("revenue", ascending=False).reset_index(drop=True)

# TODO: style with .style.bar(...) per numeric column + .set_caption(...)
styled_by_revenue = (
    product_summary.style
    .bar(subset=["revenue"], color="#5DADE2")
    .bar(subset=["profit"], color="#58D68D")
    .bar(subset=["avg_rating"], color="#F5B041", vmin=1.0, vmax=5.0)
    .format({
        "revenue": "{:,.0f}",
        "profit": "{:,.0f}",
        "avg_rating": "{:.2f}"
    })
    .set_caption("Table Lens: Product Performance Ranked by Total Revenue (In-cell bars show relative magnitude)")
)
styled_by_revenue


In [ ]:
# TODO: re-sort product_summary by profit descending and re-render the styled table
styled_by_profit = (
    product_summary.sort_values("profit", ascending=False)
    .reset_index(drop=True)
    .style
    .bar(subset=["revenue"], color="#5DADE2")
    .bar(subset=["profit"], color="#58D68D")
    .bar(subset=["avg_rating"], color="#F5B041", vmin=1.0, vmax=5.0)
    .format({
        "revenue": "{:,.0f}",
        "profit": "{:,.0f}",
        "avg_rating": "{:.2f}"
    })
    .set_caption("Table Lens: Product Performance Ranked by Total Profit")
)
styled_by_profit


**Your observation (revenue-sorted vs profit-sorted top rows):**

When re-sorting by total profit rather than total revenue, the ordering of top products changes substantially. Certain items that drive high top-line revenue (such as Smartwatches or Laptop Sleeves) generate lower net profit due to thinner profit margins, whereas high-margin categories (like Fitness or Home appliances) climb higher in the profitability ranking. For a regional manager, this stark visual difference proves that maximizing top-line sales volume does not equate to maximizing net bottom-line profit, demonstrating why multi-metric Table Lens comparisons prevent misguided inventory and promotional choices.



## Task 6 — Assemble a Single-Screen Monitoring Dashboard (open-ended)

**Real-world usage:** All three complaints at once. UrbanCart wants a **single screen** a regional
manager can glance at every morning — no interaction required to read the headline state, per the
Dashboard Patterns slide (Slide 28): critical indicators visible at a glance, with a peripheral-tier
alert that uses color (not subtlety) to flag something that needs attention.

### Your task
Build a 2x2 dashboard (`plt.subplots(2, 2, ...)`) with:

1. **Top-left — KPI summary panel** (`ax.axis("off")` + `ax.text(...)`): total revenue, total profit,
   overall average rating, and number of active stores, computed from `sales_df`.
2. **Top-right — mid-term analysis panel**: revenue by `region` (bar chart).
3. **Bottom-left — long-term trend panel**: total revenue by `month`, across all stores (line chart).
4. **Bottom-right — peripheral alert panel**: check whether **any** `(store_id, product)` pair has a
   month where `units_sold` fell below **25% of its own median** for that pair (this is the anomaly
   you found in Task 4, but write the check generically so it would catch *any* such case, not just
   the one you already know about). Show a red "ALERT — N issue(s) found" indicator if any exist, or
   a green "OK" indicator if not.

This task is intentionally the least scaffolded — you decide the exact `groupby` calls, panel layout
details, and styling. Reuse patterns from Tasks 1-5 and the Demo notebook where they fit, but write
the logic yourself.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# --- Panel 1: KPI summary (top-left) ---
# TODO
axes[0, 0].axis("off")
total_rev = sales_df["revenue"].sum()
total_prof = sales_df["profit"].sum()
overall_rating = sales_df["avg_rating"].mean()
active_stores = sales_df["store_id"].nunique()

kpi_text = (
    f"URBANCART EXECUTIVE MONITOR\n"
    f"----------------------------------------\n"
    f"Total Revenue:   Rs. {total_rev/1e6:.2f} M\n"
    f"Total Profit:    Rs. {total_prof/1e6:.2f} M ({(total_prof/total_rev)*100:.1f}% margin)\n"
    f"Avg Rating:      {overall_rating:.2f} / 5.0\n"
    f"Active Stores:   {active_stores} across 3 Regions\n"
    f"Total Catalog:   {sales_df['product'].nunique()} Products"
)
axes[0, 0].text(0.08, 0.5, kpi_text, fontsize=11, va="center", family="monospace",
                bbox=dict(boxstyle="round,pad=0.7", facecolor="#f8f9fa", edgecolor="#ced4da"))
axes[0, 0].set_title("KPI Summary (Glance-Scan Tier)", fontsize=11, fontweight="bold")

# --- Panel 2: revenue by region (top-right) ---
# TODO
region_rev = sales_df.groupby("region")["revenue"].sum().sort_values(ascending=False) / 1e6
axes[0, 1].bar(region_rev.index, region_rev.values, color=["#4C72B0", "#55A868", "#C44E52"])
axes[0, 1].set_ylabel("Revenue (Million INR)")
axes[0, 1].set_title("Mid-Term Analysis: Revenue by Region", fontsize=11, fontweight="bold")
for i, v in enumerate(region_rev.values):
    axes[0, 1].text(i, v * 0.88, f"Rs. {v:.1f}M", ha="center", color="white", fontweight="bold")

# --- Panel 3: total revenue by month, all stores (bottom-left) ---
# TODO
monthly_rev_all = sales_df.groupby("month")["revenue"].sum().reset_index()
axes[1, 0].plot(monthly_rev_all["month"], monthly_rev_all["revenue"] / 1e6, color="#2b5c8f", marker="o", markersize=4, linewidth=2)
axes[1, 0].set_ylabel("Revenue (Million INR)")
axes[1, 0].set_title("Long-Term Trend: Total Chain Revenue", fontsize=11, fontweight="bold")
axes[1, 0].tick_params(axis="x", rotation=40)

# --- Panel 4: peripheral alert panel (bottom-right) ---
# TODO: generically detect any (store_id, product) pair with a month below 25% of its own median
#       units_sold, then render a red "ALERT -- N issue(s) found" or green "OK" indicator
axes[1, 1].axis("off")
pair_medians = sales_df.groupby(["store_id", "product"])["units_sold"].transform("median")
anomalies_detected = sales_df[sales_df["units_sold"] < 0.25 * pair_medians]
n_issues = len(anomalies_detected)

if n_issues > 0:
    alert_color = "#d9534f"
    status_title = f"ALERT -- {n_issues} Anomaly Detected!"
    details = []
    for _, row in anomalies_detected.head(3).iterrows():
        details.append(f"Store {row['store_id']} | {row['product']} | {row['month'].strftime('%Y-%m')}: {row['units_sold']:.0f} units")
    alert_body = "\n".join(details)
    if n_issues > 3:
        alert_body += f"\n...and {n_issues - 3} more"
else:
    alert_color = "#5cb85c"
    status_title = "SYSTEM HEALTH: OK"
    alert_body = "All store/product pairs operating within normal median thresholds."

axes[1, 1].text(0.5, 0.72, status_title, fontsize=13, fontweight="bold", ha="center", color="white",
                bbox=dict(boxstyle="square,pad=0.5", facecolor=alert_color, edgecolor="none"))
axes[1, 1].text(0.5, 0.35, alert_body, fontsize=10, ha="center", family="monospace",
                bbox=dict(boxstyle="round,pad=0.5", facecolor="#fff3cd" if n_issues > 0 else "#e8f5e9", edgecolor=alert_color))
axes[1, 1].set_title("Continuous Peripheral Scanning (Alert Tier)", fontsize=11, fontweight="bold")

fig.suptitle("UrbanCart Regional Manager Dashboard", fontsize=14, y=0.98)
plt.tight_layout()
plt.show()


## Reflection

Answer briefly, referring back to the three manager complaints from the scenario intro:

1. **Which single technique from this activity do you think gives UrbanCart's managers the *biggest* time saving, and why?**  
   **Coordinated Multiple Views with Cross-View Brushing (Task 3)** provides the greatest time saving. It completely resolves Complaint #2 by collapsing dozens of static PDF pages into a single interactive view where selecting a store instantly synchronizes the category breakdown and historical trend, eliminating cognitive context switching and manual lookups.

2. **Which technique was hardest to implement well, and what design trade-off (from the lecture) did you have to think about while building it?**  
   The **Focus + Context drill-down (Task 4)** was the most challenging to design well. The central trade-off was managing *visual clutter vs. context loss*: plotting all 60 time series creates severe spaghetti-chart occlusion, while plotting only the anomalous line isolates the data point without giving the manager any baseline to judge whether the dip was store-specific or systemic. Blending the focus curve over a muted, aggregated context curve resolved this trade-off cleanly.

3. **If you had to add *one more* interaction technique from the lecture that this activity didn't cover (e.g. geometric fisheye, magic lens, network zooming, model-based planning), where in this retail scenario would it actually be useful, and why?**  
   An **Interactive Magic Lens** would be especially valuable. Managers could drag an on-screen movable filter lens over the dense monthly timeline or regional map to temporarily inspect secondary diagnostics (such as return rates, stockout counts, or gross margin percentages) for underlying data points without permanently cluttering or redesigning the base dashboard display.
